# Invasive (INV): OME-TIFF -> SpatialData -> precomputed raster + meshes -> Neuroglancer

Same pipeline structure as melanoma. This dataset has ~44,000 real segmented cells with confirmed `PhysicalSizeX=PhysicalSizeY=1.0 µm`

## 1. Setup

In [ ]:
from pathlib import Path
from spatialdata import SpatialData
from spatialdata.models import Labels3DModel
from dask_image.imread import imread
import xmltodict
import tifffile

from tissue_map_tools.igneous_converters import (
    from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes,
)
from tissue_map_tools.view import view_precomputed_in_vitessce

repo_root = Path.cwd().parent.parent
ome_tiff_path = repo_root / "data" / "invasive_mask.ome.tiff"
precomputed_path = repo_root / "data" / "invasive_precomputed"

if not ome_tiff_path.exists():
    raise FileNotFoundError(
        f"{ome_tiff_path} does not exist. Please use symlinks to make the data available."
    )

## 2. Load and determine axis order (same heuristic and same caveat as melanoma)

In [ ]:
data = imread(ome_tiff_path)

xml = tifffile.TiffFile(ome_tiff_path).ome_metadata
xml_dict = xmltodict.parse(xml)
sizes = {
    ax: int(xml_dict["OME"]["Image"]["Pixels"][f"@Size{ax.upper()}"]) for ax in "xyz"
}
assert len(set(sizes.values())) == 3, (
    "Sizes collide -- do not trust the heuristic below."
)

dims = []
for size in data.shape:
    for ax, ax_size in sizes.items():
        if size == ax_size:
            dims.append(ax)
            break
dims = tuple(dims)

print("data.shape =", data.shape)
print("computed dims =", dims)

## 3. Build the SpatialData object

In [ ]:
if not (precomputed_path / "info").exists():
    labels = Labels3DModel.parse(data, dims=dims)
    sdata_write_path = repo_root / "data" / "invasive_mask.zarr"

    sdata_unwritten = SpatialData.init_from_elements({"labels": labels})
    sdata_unwritten.write(str(sdata_write_path), overwrite=True)

    sdata = SpatialData.read(str(sdata_write_path))

## 4. Convert to precomputed raster + meshes

In [ ]:
%%time
import contextlib

if not (precomputed_path / "info").exists():
    stdout_path = repo_root / "data" / "invasive_stdout"
    with (
        open(stdout_path, "w") as f,
        contextlib.redirect_stdout(f),
        contextlib.redirect_stderr(f),
    ):
        from_spatialdata_raster_to_sharded_precomputed_raster_and_meshes(
            raster=sdata["labels"],
            precomputed_path=str(precomputed_path),
            shape=(128, 128, 128),
            nlod=3,
            min_chunk_size=(32, 32, 32),
        )
    print("Conversion complete.")
else:
    print("Precomputed output already exists -- skipping conversion.")

## 5. View in Neuroglancer


In [ ]:
initial_camera_state = {
    "position": [0, 0, 0],
    "projectionScale": 5466806.071355488,
    "projectionOrientation": [
        -0.636204183101654,
        -0.5028395652770996,
        0.5443811416625977,
        0.2145828753709793,
    ],
}
viewer = view_precomputed_in_vitessce(
    data_path=str(precomputed_path),
    initial_camera_state=initial_camera_state,
    segments=["2", "3", "4", "5"],
    use_web_app=True,
)
viewer